# 03 — Keep/discard log

Each section is one experiment card: **change → run → read → decide**. The decision rules are
fixed before looking at the result (they are repeated in the code). The point is not to make
every change look good; a clean **DISCARD** is a useful preliminary result.

| Card | Change | Target metric | Decision rule |
| --- | --- | --- | --- |
| C1 | broaden the offline lexicon with paraphrase synonyms | paraphrase QWK / evidence hit | KEEP if ΔQWK > 0.05 and source ΔQWK > −0.02 |
| C2 | isotonic vs ordinal calibration | MAE / ECE / QWK | KEEP BOTH if isotonic lowers MAE and ordinal is needed for probabilities |
| C3 | runtime evidence gate for abstention | selective large-error rate at 80% coverage | KEEP if gate beats confidence by > 0.02 absolute or AUC by > 0.05 |
| C4 | direct small-LLM scorer | QWK / ECE / evidence citation | KEEP AS PILOT only if it is competitive enough to justify a frontier-model run |

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from callsat_audit.config import ROOT
from callsat_audit.extract import LexiconExtractor
from callsat_audit.generate import load_datasets
from callsat_audit.llm_pipeline import direct_metrics
from callsat_audit.pipeline import (
    build_frame,
    condition_metrics,
    extract_all,
    fit_calibration,
    summarize_frame,
)

datasets = load_datasets()
source = datasets["source"]
conditions = [
    "source",
    "disfluency",
    "emotional_arc",
    "paraphrase",
    "combined",
    "persona_mix",
]

## C1 — Broaden the lexicon

**Hypothesis.** The paraphrase collapse in 02 is a span-layer failure. Adding paraphrase
synonyms to the lexicon should recover it.

**Circularity warning.** The added patterns were written by looking at the paraphrase list.
This is an oracle repair, not a deployable fix. It is useful as a diagnostic upper bound: if
even the oracle repair fails, the problem is deeper than vocabulary.

In [2]:
strict = LexiconExtractor()
broad = LexiconExtractor(broad=True)
strict_ex = {c: extract_all(datasets[c], strict, condition=c) for c in conditions}
broad_ex = {c: extract_all(datasets[c], broad, condition=c) for c in conditions}
strict_bundle = fit_calibration(source, strict_ex["source"])
broad_bundle = fit_calibration(source, broad_ex["source"])
strict_frames = {c: build_frame(datasets[c], strict_ex[c], strict_bundle) for c in conditions}
broad_frames = {c: build_frame(datasets[c], broad_ex[c], broad_bundle) for c in conditions}
for frame_dict in (strict_frames, broad_frames):
    frame_dict["source"] = frame_dict["source"][frame_dict["source"]["split"] == "holdout"].copy()

comparison = pd.DataFrame(
    {
        "strict_qwk": condition_metrics(strict_frames)["qwk_raw"],
        "broad_qwk": condition_metrics(broad_frames)["qwk_raw"],
        "strict_fact_err": {c: strict_frames[c]["fact_error_count"].mean() for c in conditions},
        "broad_fact_err": {c: broad_frames[c]["fact_error_count"].mean() for c in conditions},
        "strict_gold_hit": {c: strict_frames[c]["gold_turn_hit"].mean() for c in conditions},
        "broad_gold_hit": {c: broad_frames[c]["gold_turn_hit"].mean() for c in conditions},
    }
)
comparison["delta_qwk"] = comparison["broad_qwk"] - comparison["strict_qwk"]
comparison.round(3)

,strict_qwk,broad_qwk,strict_fact_err,broad_fact_err,strict_gold_hit,broad_gold_hit,delta_qwk
source,0.718,0.718,0.51,0.51,0.848,0.848,0.000
disfluency,0.661,0.661,0.91,0.91,0.748,0.748,0.000
emotional_arc,0.697,0.697,0.83,0.83,0.867,0.867,0.000
paraphrase,0.098,0.718,2.61,0.51,0.082,0.848,0.619
combined,0.029,0.635,2.54,0.91,0.247,0.813,0.606
persona_mix,0.782,0.782,0.48,0.48,0.833,0.833,0.000


In [3]:
target_gain = comparison.loc["paraphrase", "delta_qwk"]
source_loss = comparison.loc["source", "delta_qwk"]
c1_keep = target_gain > 0.05 and source_loss > -0.02
print(f"paraphrase delta QWK = {target_gain:+.3f}; source delta QWK = {source_loss:+.3f}")
print("C1:", "KEEP (diagnostic, circular)" if c1_keep else "DISCARD")

paraphrase delta QWK = +0.619; source delta QWK = +0.000
C1: KEEP (diagnostic, circular)


## C2 — Isotonic vs ordinal calibration

**Hypothesis.** A monotone isotonic survey map is a better point calibrator than the ordinal
model, but it cannot supply class probabilities for ECE or selective prediction.

**Decision rule.** Keep both if isotonic lowers MAE on source and the ordinal layer remains the
only source of probabilities. This is a “keep both, different jobs” decision, not a winner-take-all.

In [4]:
cal_rows = []
for condition, frame in strict_frames.items():
    latent = summarize_frame(frame, "latent")
    survey = summarize_frame(frame, "survey")
    cal_rows.append(
        {
            "condition": condition,
            "mae_raw": latent["mae_raw"],
            "mae_ordinal": latent["mae_ordinal"],
            "mae_isotonic": latent["mae_isotonic"],
            "qwk_ordinal": latent["qwk_ordinal"],
            "qwk_isotonic": latent["qwk_isotonic"],
            "ece_ordinal": latent["ece"],
            "ece_survey": survey["ece"],
            "slope_latent": latent["calibration_slope"],
        }
    )
cal = pd.DataFrame(cal_rows).set_index("condition")
cal.round(3)

,mae_raw,mae_ordinal,mae_isotonic,qwk_ordinal,qwk_isotonic,ece_ordinal,ece_survey,slope_latent
condition,,,,,,,,
source,0.625,0.631,0.624,0.710,0.717,0.050,0.128,0.985
disfluency,0.679,0.692,0.651,0.606,0.692,0.071,0.171,1.040
emotional_arc,0.652,0.666,0.668,0.663,0.697,0.065,0.103,1.018
paraphrase,0.990,1.004,0.946,0.082,0.107,0.158,0.145,1.341
combined,1.066,1.082,1.086,0.047,-0.150,0.188,0.188,0.019
persona_mix,0.668,0.637,0.630,0.720,0.743,0.063,0.087,0.979


In [5]:
iso_wins = (cal["mae_isotonic"] < cal["mae_ordinal"]).sum()
print(f"isotonic lower MAE in {iso_wins}/{len(cal)} conditions")
print("C2:", "KEEP BOTH (isotonic point, ordinal probabilities)" if iso_wins >= 3 else "REVIEW")

isotonic lower MAE in 4/6 conditions
C2: KEEP BOTH (isotonic point, ordinal probabilities)


## C3 — Runtime evidence gate

**Hypothesis.** Calls with missing resolution evidence, sentiment/resolution conflict, or low
fact confidence are more likely to be badly scored, so gating on them should beat
confidence-only abstention.

**Pre-specified gate.** `audit_score = confidence − 0.20·missing_resolution
− 0.10·conflict − 0.15·(1−span_exists) − 0.10·low_fact_confidence`.

**Decision rule.** KEEP if the gate improves the large-error rate at 80% coverage by more than
0.02 absolute, or raises AUC by more than 0.05.

In [6]:
def add_gate(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["audit_gate"] = (
        out["confidence"]
        - 0.20 * out["missing_resolution_evidence"]
        - 0.10 * out["sentiment_resolution_conflict"]
        - 0.15 * (1.0 - out["span_exists"])
        - 0.10 * (out["fact_confidence"] < 0.55).astype(float)
    )
    out["large_error"] = (np.abs(out["pred_label"] - out["latent"]) > 1).astype(int)
    return out


def selective_rate(frame: pd.DataFrame, score_col: str, coverage: float = 0.80) -> float:
    n = int(len(frame) * coverage)
    order = np.argsort(-frame[score_col].to_numpy())[:n]
    return float(frame["large_error"].to_numpy()[order].mean())


gate_rows = []
for condition in ("source", "disfluency", "emotional_arc", "paraphrase"):
    frame = add_gate(strict_frames[condition])
    gate_rows.append(
        {
            "condition": condition,
            "auc_confidence": roc_auc_score(frame["large_error"], frame["confidence"]),
            "auc_gate": roc_auc_score(frame["large_error"], frame["audit_gate"]),
            "large_error_conf_80": selective_rate(frame, "confidence", 0.80),
            "large_error_gate_80": selective_rate(frame, "audit_gate", 0.80),
        }
    )
gate = pd.DataFrame(gate_rows).set_index("condition")
gate["delta_auc"] = gate["auc_gate"] - gate["auc_confidence"]
gate["delta_large_error"] = gate["large_error_gate_80"] - gate["large_error_conf_80"]
gate.round(3)

,auc_confidence,auc_gate,large_error_conf_80,large_error_gate_80,delta_auc,delta_large_error
condition,,,,,,
source,0.372,0.387,0.075,0.075,0.015,0.000
disfluency,0.275,0.264,0.088,0.062,-0.010,-0.025
emotional_arc,0.354,0.322,0.088,0.075,-0.032,-0.012
paraphrase,0.478,0.443,0.262,0.288,-0.035,0.025


In [7]:
# The first pass used an "any condition" rule. It fired on a single 2.5 pp cell while
# losing AUC elsewhere, so we require a consistent improvement instead. This is recorded
# as a protocol deviation in EXPERIMENT_LOG.md.
consistent_gain = int((gate["delta_large_error"] < -0.005).sum())
consistent_loss = int((gate["delta_large_error"] > 0.005).sum())
c3_keep = consistent_gain >= 3 and consistent_loss == 0
print(
    f"conditions improved: {consistent_gain}/{len(gate)}; "
    f"conditions worsened: {consistent_loss}/{len(gate)}"
)
print("C3:", "KEEP" if c3_keep else "DISCARD (not consistent across conditions)")

conditions improved: 2/4; conditions worsened: 1/4
C3: DISCARD (not consistent across conditions)


## C4 — Direct small-LLM scoring pilot

The direct arm asks a local Qwen2.5-3B-Instruct model for a 1–5 score, a five-class
distribution, verbalised confidence, cited turns, and a quote. It is scored on the same 40
source-holdout calls as the symbolic arm and their paraphrase twins. This is a **feasibility
probe**, not a model comparison: one small open model, 40 calls per condition, temperature 0.

The audit question is whether the direct arm's citations and confidence are trustworthy enough
to support the same kind of evidence-chain audit. If not, the evidence-chain arm remains the
only auditable option in this pilot.

In [8]:
llm_metrics_path = ROOT / "results" / "runs" / "llm_pilot_direct_metrics.csv"
if llm_metrics_path.exists():
    llm_metrics = pd.read_csv(llm_metrics_path, index_col=0)
    display_cols = [
        "n",
        "qwk",
        "mae",
        "ece_probs",
        "ece_confidence",
        "mean_confidence",
        "cited_in_gold_rate",
        "quote_support_rate",
        "parse_rate",
        "mean_latency_s",
    ]
    display(llm_metrics[[c for c in display_cols if c in llm_metrics.columns]].round(3))
else:
    print("LLM pilot metrics not found; run scripts/run_llm_pilot.py first.")

,n,qwk,mae,ece_probs,ece_confidence,mean_confidence,cited_in_gold_rate,quote_support_rate,parse_rate,mean_latency_s
condition,,,,,,,,,,
source,40.0,0.669,0.8,0.575,0.498,0.874,0.975,0.95,1.000,3.223
paraphrase,40.0,0.667,0.8,0.593,0.478,0.854,0.925,0.90,0.975,3.304


In [9]:
# Paired comparison on the exact LLM call ids, so the two arms see the same calls.
llm_source_path = ROOT / "results" / "runs" / "llm_pilot_direct_source.csv"
if llm_metrics_path.exists() and llm_source_path.exists():
    llm_source = pd.read_csv(llm_source_path)
    ids = set(llm_source["call_id"])
    sym_rows = []
    for condition in ("source", "disfluency", "emotional_arc", "paraphrase"):
        subset = strict_frames[condition][strict_frames[condition]["call_id"].isin(ids)]
        if len(subset) == 0:
            continue
        metrics = summarize_frame(subset, "latent")
        sym_rows.append(
            {
                "condition": condition,
                "symbolic_qwk": metrics["qwk_raw"],
                "symbolic_ece": metrics["ece"],
                "symbolic_large_error": metrics["large_error_rate"],
            }
        )
    symbolic_subset = pd.DataFrame(sym_rows).set_index("condition")
    comparison = symbolic_subset.join(
        llm_metrics[["qwk", "ece_probs", "large_error_rate"]].rename(
            columns={
                "qwk": "llm_qwk",
                "ece_probs": "llm_ece",
                "large_error_rate": "llm_large_error",
            }
        ),
        how="inner",
    )
    display(comparison.round(3))
    evidence = llm_metrics.loc[
        [c for c in comparison.index if c in llm_metrics.index],
        ["cited_in_gold_rate", "quote_support_rate", "ece_confidence"],
    ]
    display(evidence.round(3))
else:
    print("Run the LLM pilot and keep its per-call CSV to enable the paired comparison.")

,symbolic_qwk,symbolic_ece,symbolic_large_error,llm_qwk,llm_ece,llm_large_error
condition,,,,,,
source,0.746,0.090,0.20,0.669,0.575,0.15
paraphrase,0.113,0.199,0.45,0.667,0.593,0.15


,cited_in_gold_rate,quote_support_rate,ece_confidence
condition,,,
source,0.975,0.95,0.498
paraphrase,0.925,0.90,0.478


### Reading C4

The comparison that matters is **source vs paraphrase**. If the direct LLM holds its QWK under
paraphrase while the symbolic evidence chain collapses (0.098), the lexical failure is an
**extractor artefact**, not a property of the calibration layer. The cost is visible in the
same table: confidence ECE 0.48–0.59, mean confidence 0.85–0.87, and citation precision
0.53–0.60. The direct model is more robust but less auditable; the evidence-chain model is
more auditable but brittle to vocabulary. The next design should put a *robust* extractor
inside the auditable chain, not choose one arm over the other.

## Final ledger

The table below is the artefact to carry into the paper draft. It records what was tried, what
moved, and what was rejected. The prose summary lives in `EXPERIMENT_LOG.md`.

**Protocol deviation (C3).** The initial decision rule was “keep if any condition improves
by 2 pp or 0.05 AUC”. It fired on a single disfluency cell (−2.5 pp) while the gate lost AUC
there and worsened paraphrase by +2.5 pp. We tightened the rule to require improvement in at
least three of four conditions with no material regression, and recorded **DISCARD**. The raw
per-condition table is left visible so a reader can apply their own rule.

In [10]:
ledger = pd.DataFrame(
    [
        {
            "card": "C0",
            "change": "strict lexicon + ordinal/isotonic source calibration (baseline)",
            "evidence": "02 stress profile",
            "decision": "REFERENCE",
            "note": "source QWK is the optimistic ceiling for this DGP",
        },
        {
            "card": "C1",
            "change": "broaden lexicon with paraphrase synonyms",
            "evidence": f"paraphrase ΔQWK={target_gain:+.3f}, source ΔQWK={source_loss:+.3f}",
            "decision": "KEEP (diagnostic, circular)" if c1_keep else "DISCARD",
            "note": "lexicon was written from the paraphrase list; not a deployable fix",
        },
        {
            "card": "C2",
            "change": "isotonic point calibrator alongside ordinal probabilities",
            "evidence": f"isotonic lower MAE in {iso_wins}/{len(cal)} conditions",
            "decision": "KEEP BOTH" if iso_wins >= 3 else "REVIEW",
            "note": "isotonic for point scores; ordinal for ECE/selective/conformal",
        },
        {
            "card": "C3",
            "change": "runtime evidence gate for abstention",
            "evidence": (
                f"improved {consistent_gain}/{len(gate)}; "
                f"worsened {consistent_loss}/{len(gate)}; "
                f"max ΔAUC={gate['delta_auc'].max():+.3f}"
            ),
            "decision": "KEEP" if c3_keep else "DISCARD",
            "note": "presence flags are diagnostic but not a consistent deferral policy",
        },
        {
            "card": "C4",
            "change": "direct small-LLM scoring pilot",
            "evidence": "LLM table above",
            "decision": "PILOT ONLY",
            "note": "one 3B local model, 40 source calls + paraphrase twins; not a frontier-LLM result",
        },
    ]
)
ledger.to_csv(ROOT / "results" / "runs" / "03_ledger.csv", index=False)
ledger

,card,change,evidence,decision,note
0,C0,strict lexicon + ordinal/isotonic source calib...,02 stress profile,REFERENCE,source QWK is the optimistic ceiling for this DGP
1,C1,broaden lexicon with paraphrase synonyms,"paraphrase ΔQWK=+0.619, source ΔQWK=+0.000","KEEP (diagnostic, circular)",lexicon was written from the paraphrase list; ...
2,C2,isotonic point calibrator alongside ordinal pr...,isotonic lower MAE in 4/6 conditions,KEEP BOTH,isotonic for point scores; ordinal for ECE/sel...
3,C3,runtime evidence gate for abstention,improved 2/4; worsened 1/4; max ΔAUC=+0.015,DISCARD,presence flags are diagnostic but not a consis...
4,C4,direct small-LLM scoring pilot,LLM table above,PILOT ONLY,"one 3B local model, 40 source calls + paraphra..."


## What the next iteration should test

1. **Replace the symbol oracle with a real LLM extractor** on the same 100-call holdouts, with
   inter-model agreement as an additional audit signal. The 3B pilot is too weak to settle this.
2. **Make the audit semantic, not presence-based.** The failure of C3 suggests that “no evidence”
   is often the scorer's honest default, while confident wrong evidence is the dangerous case.
   A contradiction check between cited span and typed fact is the next candidate.
3. **Test calibration under label-preserving shift only.** Emotional arc changes the construct;
   it should be analysed as concept shift, not as a calibration failure without qualification.
4. **Pre-register the tolerance for score shifts.** The 0.5 raw-score tolerance is coarse for
   sentiment-only changes; report label-change and expected-score deltas alongside it.